# Gemma 3 12B Legal AI — Fine-Tuning & Export Notebook

**Model**: `google/gemma-3-12b-it` → `gemma3-legal` (LoRA fine-tuned)
**Hardware**: NVIDIA A100 40GB (Colab) → RTX 3060 Ti 8GB (deployment)
**Quantization**: 4-bit (training) → INT4 AWQ (TRT-LLM) or Q4_K_M (GGUF/Ollama)

---

## GOLDEN RULES

```
1. ALWAYS save BEFORE calling for_inference()
   for_inference() activates torch.compile / FakeTensorMode.
   ONE-WAY DOOR — cannot save after. Even del model won't fix it.
   Only a full runtime restart clears it.

   ORDER: Load → Merge LoRA → Save ALL exports → THEN for_inference()

2. MUST call merge_and_unload() BEFORE merged/GGUF saves
   PeftModel.from_pretrained() wraps the model, but Unsloth's
   save_pretrained_merged() and save_pretrained_gguf() don't
   detect it. They save the BASE model without your training.
   Call model = model.merge_and_unload() first to fuse LoRA
   weights permanently into the base model.
   
   WARNING SIGN: "Model is not a PeftModel (no Lora adapters detected)"
   = your save does NOT contain legal fine-tuning!

3. Use google/gemma-3-12b-it (NOT unsloth pre-quantized) for reload
   The unsloth/gemma-3-12b-it-bnb-4bit version can have quant_state
   mismatches with newer bitsandbytes versions.

4. Chat template: content must be list-of-dicts, NOT plain string
   {"role": "user", "content": [{"type": "text", "text": prompt}]}

5. TRT-LLM engines built on A100 (SM 80) run on RTX 3060 Ti (SM 86)
   Both are Ampere. SM 80 PTX forward-compiles to SM 86 via JIT.

6. GGUF save needs config.json — save to LOCAL disk first, copy to Drive
   save_pretrained_gguf() to an empty Drive folder fails with
   "config.json does not exist". Always save locally, then copy.
```

---
## 1. Install Dependencies

In [ ]:
!pip install unsloth
!pip install --upgrade pillow

# Verify GPU
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

---
## 2. Mount Google Drive & HuggingFace Login

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import shutil, os, time

# HuggingFace login (required for google/gemma-3-12b-it)
# Set HF_TOKEN in Colab Secrets (key icon in sidebar) for auto-login
from huggingface_hub import login
login()  # uses Colab Secrets or prompts for token

# Paths — edit these if your Drive layout differs
DRIVE_BASE = Path("/content/drive/MyDrive")
CHECKPOINT_DIR = DRIVE_BASE / "gemma3-legal-checkpoint-3750"
LORA_DIR = DRIVE_BASE / "gemma3-legal-lora"
MERGED_DIR = DRIVE_BASE / "gemma3-legal-merged-16bit"
GGUF_DIR = DRIVE_BASE / "gemma3-legal-gguf"
AWQ_DIR = DRIVE_BASE / "gemma3-legal-awq-int4"
ENGINE_DIR = DRIVE_BASE / "gemma3-legal-trtllm-engine"

print(f"Checkpoint exists: {CHECKPOINT_DIR.exists()}")
if CHECKPOINT_DIR.exists():
    for f in sorted(CHECKPOINT_DIR.iterdir()):
        sz = f.stat().st_size / 1024 / 1024
        print(f"  {f.name}  ({sz:.1f} MB)")

---
## 3. Load Base Model + LoRA Adapter

Two-step load: official base model with fresh quantization,
then LoRA adapter applied on top.

In [ ]:
from unsloth import FastVisionModel

# Step 1: Load base model with FRESH quantization
model, tokenizer = FastVisionModel.from_pretrained(
    "google/gemma-3-12b-it",
    load_in_4bit=True,
)
print("Base model loaded with fresh quantization.")

# Step 2: Apply YOUR trained LoRA adapter
from peft import PeftModel
model = PeftModel.from_pretrained(
    model,
    str(CHECKPOINT_DIR),
)
print(f"LoRA adapter loaded from {CHECKPOINT_DIR}")
print("\nModel ready! DO NOT call for_inference() yet — SAVE FIRST!")

---
## 4. Save Exports (DO THIS BEFORE for_inference!)

**Critical order:**
1. Save LoRA adapter (4A) — fast safety backup
2. Merge LoRA into base model (4B) — fuses weights permanently
3. Save merged 16-bit (4C) — for TRT-LLM/AWQ conversion
4. Save GGUF (4D) — for Ollama deployment
5. Copy to Drive (4E) — backup merged shards

### 4A. Save LoRA Adapter (~261 MB, fastest safety backup)

In [ ]:
LORA_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))
print(f"LoRA adapter saved to {LORA_DIR}")

### 4B. Merge LoRA into Base Model

**MUST run this before merged/GGUF saves.**
Without this step, saves contain the base model WITHOUT legal training.

Warning sign: `"Model is not a PeftModel"` = save has NO legal fine-tuning!

In [ ]:
# Fuse LoRA weights permanently into base model
print("Merging LoRA adapter into base model...")
model = model.merge_and_unload()
print("LoRA merged! Model now contains legal fine-tuning in base weights.")
print("Note: model is no longer a PeftModel — LoRA adapter save (4A) still valid.")

### 4C. Save Merged 16-bit (~24 GB, needed for TRT-LLM + AWQ)

In [ ]:
stat = shutil.disk_usage("/content")
free_gb = stat.free / (1024**3)
print(f"Disk free: {free_gb:.1f} GB (need ~24 GB for merged shards)")

if free_gb < 25:
    print("WARNING: Low disk space! Skip merged, use LoRA + GGUF instead.")
else:
    LOCAL_MERGED = Path("gemma3-legal-merged-16bit")
    LOCAL_MERGED.mkdir(parents=True, exist_ok=True)

    # Save with HF's save_pretrained (model is already merged)
    model.save_pretrained(str(LOCAL_MERGED))
    tokenizer.save_pretrained(str(LOCAL_MERGED))

    total_gb = sum(f.stat().st_size for f in LOCAL_MERGED.iterdir()) / 1024**3
    print(f"Merged 16-bit saved locally to {LOCAL_MERGED} ({total_gb:.1f} GB)")
    for f in sorted(LOCAL_MERGED.glob("*.safetensors")):
        print(f"  {f.name} ({f.stat().st_size / 1024**3:.2f} GB)")
    print("Run 4E to copy to Google Drive.")

### 4D. Save GGUF Q4_K_M (~7 GB, for Ollama deployment)

Saves locally first (GGUF converter needs config.json in the directory),
then copies the .gguf file to Google Drive.

In [ ]:
import shutil
from pathlib import Path

LOCAL_MERGED = Path("gemma3-legal-merged-16bit")
LOCAL_GGUF   = Path("gemma3-legal-gguf-local")

# Verify merged model exists
if not LOCAL_MERGED.exists() or not (LOCAL_MERGED / "config.json").exists():
    raise FileNotFoundError(
        "No merged model found locally. Run Cell 4C first!\n"
        f"Expected: {LOCAL_MERGED}/config.json"
    )

# Create GGUF output dir and pre-copy required files
LOCAL_GGUF.mkdir(exist_ok=True, parents=True)

# Step 1: Copy config + tokenizer files (GGUF converter needs these in output dir)
config_files = [
    "config.json", "tokenizer.json", "tokenizer_config.json",
    "chat_template.jinja", "generation_config.json",
    "special_tokens_map.json", "tokenizer.model",
]
copied = 0
for f in config_files:
    src = LOCAL_MERGED / f
    if src.exists():
        shutil.copy2(str(src), str(LOCAL_GGUF / f))
        copied += 1
        print(f"  Copied {f}")
print(f"Config files: {copied} copied to {LOCAL_GGUF}/")

# Step 2: Copy safetensors (converter needs model weights too)
for f in LOCAL_MERGED.glob("*.safetensors"):
    dst = LOCAL_GGUF / f.name
    if not dst.exists():
        print(f"  Copying {f.name} ({f.stat().st_size / 1e9:.2f} GB)...")
        shutil.copy2(str(f), str(dst))
    else:
        print(f"  {f.name} already exists, skipping")

# Step 3: Run GGUF conversion (~16 min)
print("\nStarting GGUF Q4_K_M conversion...")
model.save_pretrained_gguf(
    str(LOCAL_GGUF),
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF conversion complete!")

# Step 4: Copy .gguf files to Google Drive
GGUF_DIR = Path("/content/drive/MyDrive/gemma3-legal-finetuned/gguf")
GGUF_DIR.mkdir(parents=True, exist_ok=True)

gguf_files = list(LOCAL_GGUF.glob("*.gguf"))
if not gguf_files:
    print("WARNING: No .gguf files found! Conversion may have failed.")
else:
    for f in gguf_files:
        print(f"  Copying {f.name} ({f.stat().st_size / 1e9:.2f} GB) to Drive...")
        shutil.copy2(str(f), str(GGUF_DIR / f.name))
    print(f"\nDone! {len(gguf_files)} GGUF file(s) saved to {GGUF_DIR}")

### 4E. Copy Merged Shards to Google Drive (resumable)

Re-run safe — already-copied shards are skipped by size check.

In [ ]:
LOCAL_MERGED = Path("gemma3-legal-merged-16bit")
if not LOCAL_MERGED.exists():
    print("No merged model found locally. Run 4C first.")
else:
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    all_files = sorted(LOCAL_MERGED.iterdir())

    print(f"Copying {len(all_files)} files to Drive (skips existing)...\n")
    copied, skipped = 0, 0

    for i, src in enumerate(all_files, 1):
        dst = MERGED_DIR / src.name
        src_size = src.stat().st_size
        size_gb = src_size / (1024**3)

        if dst.exists() and dst.stat().st_size == src_size:
            print(f"  [{i}/{len(all_files)}] SKIP {src.name} ({size_gb:.2f} GB)")
            skipped += 1
            continue

        print(f"  [{i}/{len(all_files)}] {src.name} ({size_gb:.2f} GB)...", end=" ", flush=True)
        t0 = time.time()
        shutil.copy2(str(src), str(dst))
        elapsed = time.time() - t0
        speed = src_size / elapsed / (1024**2) if elapsed > 0 else 0
        print(f"done ({elapsed:.0f}s, {speed:.0f} MB/s)")
        copied += 1

    print(f"\nDone: {copied} copied, {skipped} skipped")
    print(f"If Colab disconnects mid-copy, just re-run this cell.")

### 4F. Download to Desktop

In [ ]:
from google.colab import files

# Download LoRA adapter (small, fast)
shutil.make_archive("/content/gemma3-legal-lora", 'zip', str(LORA_DIR))
files.download("/content/gemma3-legal-lora.zip")

In [ ]:
# Download GGUF (single file, Ollama-ready)
gguf_files = list(GGUF_DIR.glob("*.gguf"))
if gguf_files:
    files.download(str(gguf_files[0]))
else:
    print("No GGUF file found on Drive. Run 4D first.")

---
## 5. Inference Testing (AFTER saves are complete)

**POINT OF NO RETURN** — all saves must be done before this cell.
If you need to save again after this, you MUST restart the runtime.

In [ ]:
# ============================================
# POINT OF NO RETURN — all saves must be done!
# ============================================
FastVisionModel.for_inference(model)
print("Inference mode enabled. Cannot save after this.")

In [ ]:
from transformers import TextStreamer

def ask(prompt, system="You are a legal AI assistant specialized in evidence analysis, case law, and legal document processing.", max_tokens=512):
    """Send a properly-formatted chat prompt to the model.
    
    IMPORTANT: content must be list-of-dicts format for vision models.
    [{"type": "text", "text": prompt}] NOT just a plain string.
    Plain string causes: TypeError: string indices must be integers
    """
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system}]},
        {"role": "user", "content": [{"type": "text", "text": prompt}]},
    ]

    input_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False,
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"Prompt: {prompt}")
    print(f"{'='*60}")

    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_tokens,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        use_cache=True,
    )
    print()

In [ ]:
# Legal domain test queries
ask("Explain evidence chain of custody requirements for digital forensics.")
ask("What are the key elements of a RAG pipeline for legal document search?")
ask("Describe the Svelte 5 runes system and how $state() differs from writable stores.")

---
## 6. Ollama Deployment (Local RTX 3060 Ti)

After downloading the GGUF file to your local machine:

```bash
# Create Modelfile
cat > Modelfile <<'EOF'
FROM ./unsloth.Q4_K_M.gguf
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
SYSTEM "You are a legal AI assistant specialized in evidence analysis, case law, and legal document processing."
EOF

# Create the Ollama model
ollama create gemma3-legal-v2 -f Modelfile

# Test it
ollama run gemma3-legal-v2 "What is chain of custody?"
```

---
## 7. TRT-LLM INT4 Engine (for Triton Inference Server)

### Architecture: Build on Colab, Run on RTX 3060 Ti

TensorRT engines use PTX (Parallel Thread Execution) intermediate
representation. SM 80 (A100) PTX forward-compiles to SM 86 (RTX 3060 Ti)
via the GPU driver's JIT compiler at runtime. Both are Ampere family.

```
Colab A100 (SM 80)          Local RTX 3060 Ti (SM 86)
─────────────────           ────────────────────────
AWQ INT4 quantize           
    ↓                       
trtllm-build                
    ↓                       
engine.plan (PTX)           
    ↓ Google Drive           
    ─────────────────→      Triton loads engine
                            Driver JIT: PTX → SM 86 SASS
                            ~30-60s first load (cached after)
                            ~10% perf penalty vs native build
```

### 7A. AWQ INT4 Quantization (on Colab — needs VRAM)

In [ ]:
# ============================================
# AWQ INT4 quantization for TRT-LLM
# Requires: merged 16-bit model from step 4C
# ============================================

# Uncomment to run:
"""
!pip install autoawq
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive")
merged_path = str(DRIVE_BASE / "gemma3-legal-merged-16bit")
awq_path = str(DRIVE_BASE / "gemma3-legal-awq-int4")

print("Loading merged 16-bit model for AWQ quantization...")
model_awq = AutoAWQForCausalLM.from_pretrained(merged_path)
tok_awq = AutoTokenizer.from_pretrained(merged_path)

quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM",
}

print("Quantizing to INT4 AWQ (this takes ~30 min)...")
model_awq.quantize(tok_awq, quant_config=quant_config)
model_awq.save_quantized(awq_path)
tok_awq.save_pretrained(awq_path)
print(f"AWQ INT4 saved to {awq_path}")
"""
print("Uncomment the code above to run AWQ quantization.")
print("Requires merged 16-bit model from step 4C.")

### 7B. TRT-LLM Engine Build (on Colab — portable to RTX 3060 Ti)

Both A100 (SM 80) and RTX 3060 Ti (SM 86) are Ampere.
SM 80 PTX forward-compiles to SM 86 via driver JIT.

In [ ]:
# ============================================
# TRT-LLM engine build on Colab
# Engine uses PTX — portable across Ampere GPUs
# SM 80 (A100) → JIT to SM 86 (RTX 3060 Ti)
# ============================================

# Uncomment to run:
"""
!pip install tensorrt_llm

import subprocess
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive")
AWQ_DIR = DRIVE_BASE / "gemma3-legal-awq-int4"
ENGINE_DIR = DRIVE_BASE / "gemma3-legal-trtllm-engine"
CKPT_DIR = Path("/tmp/gemma3_trt_ckpt")

ENGINE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Convert AWQ checkpoint to TRT-LLM format
print("Converting AWQ checkpoint...")
subprocess.run([
    "python3", "-m", "tensorrt_llm.commands.convert_checkpoint",
    "--model_dir", str(AWQ_DIR),
    "--output_dir", str(CKPT_DIR),
    "--dtype", "float16",
    "--use_weight_only",
    "--weight_only_precision", "int4_awq",
    "--tp_size", "1",
    "--pp_size", "1",
], check=True)

# Step 2: Build TRT-LLM engine (PTX embedded — portable to SM 86)
print("Building TRT-LLM engine (this takes 20-40 min)...")
subprocess.run([
    "trtllm-build",
    "--checkpoint_dir", str(CKPT_DIR),
    "--output_dir", str(ENGINE_DIR),
    "--gemm_plugin", "float16",
    "--max_batch_size", "4",
    "--max_input_len", "2048",
    "--max_seq_len", "4096",
    "--paged_kv_cache", "enable",
    "--use_paged_context_fmha", "enable",
    "--max_num_tokens", "8192",
], check=True)

print(f"Engine saved to {ENGINE_DIR}")
print("Download to local machine and run in Triton.")
print("First load on RTX 3060 Ti: ~30-60s JIT warmup (cached after).")
"""
print("Uncomment the code above to build TRT-LLM engine.")
print("Requires AWQ INT4 model from step 7A.")

### 7C. Triton Deployment (Local RTX 3060 Ti)

After downloading the engine from Google Drive:

#### Model Repository Structure
```
triton-models/
└── gemma3_legal/
    ├── config.pbtxt
    └── 1/
        └── model.plan          # TRT-LLM engine from 7B
```

#### config.pbtxt
```protobuf
name: "gemma3_legal"
backend: "tensorrtllm"
max_batch_size: 4

input [
  { name: "input_ids", data_type: TYPE_INT32, dims: [-1] },
  { name: "input_lengths", data_type: TYPE_INT32, dims: [1] },
  { name: "request_output_len", data_type: TYPE_INT32, dims: [1] }
]

output [
  { name: "output_ids", data_type: TYPE_INT32, dims: [-1, -1] },
  { name: "sequence_length", data_type: TYPE_INT32, dims: [-1] }
]

instance_group [{ count: 1, kind: KIND_GPU, gpus: [0] }]

parameters {
  key: "max_beam_width"
  value: { string_value: "1" }
}
parameters {
  key: "gpt_model_type"
  value: { string_value: "inflight_fused_batching" }
}
```

#### docker-compose.triton-gemma3.yml
```yaml
services:
  triton-gemma3:
    image: nvcr.io/nvidia/tritonserver:24.10-trtllm-python-py3
    container_name: triton-gemma3-legal
    restart: unless-stopped
    ports:
      - "8099:8000"   # HTTP — matches existing TRTLLMClient
      - "8098:8001"   # gRPC
      - "8097:8002"   # Metrics (Prometheus)
    volumes:
      - ./triton-models:/models:ro
    environment:
      - CUDA_VISIBLE_DEVICES=0
    deploy:
      resources:
        limits:
          memory: 7G
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    command: >
      tritonserver
      --model-repository=/models
      --strict-model-config=false
      --backend-config=tensorrtllm,batching_type=inflight_fused_batching
      --backend-config=tensorrtllm,max_queue_delay_microseconds=1000
      --log-verbose=1
      --metrics-port=8002
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/v2/health/ready"]
      interval: 30s
      timeout: 10s
      retries: 5
      start_period: 120s
```

#### SvelteKit Integration
Your existing `TRTLLMClient` at `src/lib/trt-llm/client.ts` already
targets port 8099 via `PUBLIC_TRT_LLM_ENDPOINT`. Update API paths
to Triton v2:
```
POST /v2/models/gemma3_legal/generate          → non-streaming
POST /v2/models/gemma3_legal/generate_stream    → SSE streaming
GET  /v2/health/ready                           → health check
```

#### VRAM Budget (RTX 3060 Ti 8GB)
| Component | VRAM |
|-----------|------|
| Gemma3 12B INT4 AWQ | ~5.5 GB |
| KV cache (paged, 4096 seq) | ~1.5 GB |
| Overhead | ~0.5 GB |
| **Total** | **~7.5 GB** |

---
## 8. Emergency: Save Checkpoint Files Directly

If the model is corrupted (FakeTensorMode, etc.) and you can't
save through the model API, just copy the raw files from disk.
The `.safetensors` files ARE your trained weights.

In [ ]:
# Emergency file copy — no model loading needed
# Use this when model.save_pretrained() fails
"""
import shutil, os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Find your checkpoint directory
ckpt_dir = "./gemma3-12b-legal-outputs/checkpoint-3750"  # adjust path
dst = "/content/drive/MyDrive/gemma3-legal-checkpoint-3750"

print("Checkpoint contents:")
for f in sorted(os.listdir(ckpt_dir)):
    sz = os.path.getsize(os.path.join(ckpt_dir, f)) / 1024 / 1024
    print(f"  {f}  ({sz:.1f} MB)")

os.makedirs(dst, exist_ok=True)
shutil.copytree(ckpt_dir, dst, dirs_exist_ok=True)
print(f"\nCheckpoint copied to {dst}")
"""

---
## Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `FakeTensor ... aten.detach` | Called `for_inference()` before save | Restart runtime, reload, save first |
| `Params4bit ... FakeTensor` | FakeTensorMode persists after `del model` | Full runtime restart required |
| `"not a PeftModel"` warning | Unsloth doesn't detect PeftModel wrapper | Run `model.merge_and_unload()` before save |
| `config.json does not exist` | GGUF saved to empty Drive folder | Save locally first, copy .gguf to Drive |
| `string indices must be integers` | Chat template expects list-of-dicts | Use `[{"type": "text", "text": prompt}]` not plain string |
| `Skipping ... no quant_state found` | bitsandbytes version mismatch | Use `google/gemma-3-12b-it` + `load_in_4bit=True` |
| Model outputs nonsense/math | Base model weights bad or LoRA not merged | Two-step load + `merge_and_unload()` before save |
| `attention_mask not set` | pad_token == eos_token | Pass `attention_mask` or ignore |
| Low disk for merged save | 24 GB needed for 16-bit shards | Save LoRA + GGUF instead, merge later |
| `NameError: model not defined` | `del model` was called earlier | Restart runtime, reload from checkpoint |
| TRT-LLM engine won't load on different GPU | Engine built for specific SM arch | Both GPUs must be Ampere+ for PTX JIT |